In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
from openai import OpenAI
import os
openai_client = OpenAI(
    api_key=os.getenv("OPEN_API_KEY")
)

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

In [5]:
question="I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—possibly, but it depends on the course rules and whether registration is still open.

A few quick things to check:
- **Enrollment deadline**: Some courses allow late joining for the first week or two.
- **Access to materials**: You may still be able to get the lessons and recordings even if you missed the start.
- **Assignments/exams**: Ask whether you can complete missed work or get an adjusted schedule.
- **Seat availability**: If it’s a live or limited-capacity course, there may be a waitlist.

If you want, I can help you write a short message to the course organizer asking if you can join now.


In [6]:
context = '''
General Course-Related Questions
#
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
edit on GitHub
#
Cloud alternatives with GPU

Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

    Google Colab
    Kaggle
    Databricks (possibly)

Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
edit on GitHub
#
Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

    First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
    Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!

'''

In [7]:
prompt = f'''
your taks is to answer the question based on the following context.
Use the context to find relevant information to answer the question and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".

Question: {question}

Context: {context}
'''

In [8]:
print(prompt)


your taks is to answer the question based on the following context.
Use the context to find relevant information to answer the question and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".

Question: I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
#
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom lin

In [9]:
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.


In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [11]:
import requests
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw[:2]

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93}]

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'''{url_prefix}/{course['path']}'''
    
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)

1342

In [14]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
from minsearch import Index
index = Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [22]:
import pandas as pd
pd.DataFrame(documents)['course'].unique()

<StringArray>
[       'data-engineering-zoomcamp', 'stock-markets-analytics-zoomcamp',
            'ai-dev-tools-zoomcamp',                     'llm-zoomcamp',
                   'mlops-zoomcamp',        'machine-learning-zoomcamp']
Length: 6, dtype: str

In [34]:
question = 'I just discovered the course. Can I join now?'
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2, 'section':0.5}
    filter_dict = {'course':course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results =5
    )


In [36]:
search_results = search(question, course='llm-zoomcamp')
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [1]:
INSTRUCTIONS = '''
Your task is to answer the questions from the course partipants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,
respond with "I don't know".
'''

In [2]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []
    
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q:")